
# MCP Management with LlamaStack

This notebook will walk you through the process of managing Model Context Protocol (MCP) toolgroups using the `llama-stack-client`. It provides a hands-on approach to understanding how MCP enables tools to be dynamically registered and invoked in a LlamaStack-based system. 

###  What is MCP?

MCP tools are special tools that can interact with llama stack over model context protocol. These tools are dynamically discovered from an MCP endpoint and can be used to extend the agent’s capabilities. 

MCP tools require:

- A valid MCP endpoint URL.
- The endpoint must implement the Model Context Protocol.
- Tools are discovered dynamically from the endpoint.

## Prerequisites 

Before starting, ensure:

- You have a running instance of a LlamaStack server (local or remote)
- You have configured `.env` with the following variables:

```env
REMOTE_BASE_URL=http://your-llamastack-url
MCP_SERVER_URL=http://your-mcp-url
```

For detailed llama-stack server setup instructions, refer to:

- [Remote Setup Guide](https://github.com/opendatahub-io/llama-stack-on-ocp/tree/main/kubernetes)
- [Local Setup Guide](https://github.com/redhat-et/agent-frameworks/tree/main/prototype/frameworks/llamastack)

In [38]:
import os
from llama_stack_client import LlamaStackClient
from dotenv import load_dotenv
load_dotenv()

base_url=os.getenv("REMOTE_BASE_URL")
# connecting to remote server
client = LlamaStackClient(base_url=base_url)

## View Registered tool groups

Refer to [Link](https://github.com/opendatahub-io/llama-stack-demos/tree/main/mcp-servers), how different servers are defined and deployed on OpenShift. Use `client.tools.list()` to inspect which toolgroups are currently registered with the LlamaStack server. 

In [39]:
toolgroups = client.tools.list()
registered_toolgroups = set(t.toolgroup_id for t in toolgroups)

print("Registered Toolgroups:")
for tg in registered_toolgroups:
    print(f" - {tg}")

Registered Toolgroups:
 - mcp::openshift
 - builtin::websearch
 - mcp::custom_tool
 - mcp::github
 - builtin::code_interpreter
 - builtin::rag
 - mcp::ansible


##  View Tools in a Toolgroup

Let’s examine the tools available in a specific toolgroup like `mcp::openshift`. 

In [40]:
def list_tools_in_group(toolgroup_id):
    tools = client.tools.list(toolgroup_id=toolgroup_id)
    return [t.identifier for t in tools]

In [41]:
# Example
list_tools_in_group("mcp::openshift")

['configuration_view',
 'events_list',
 'namespaces_list',
 'pods_delete',
 'pods_exec',
 'pods_get',
 'pods_list',
 'pods_list_in_namespace',
 'pods_log',
 'pods_run',
 'projects_list',
 'resources_create_or_update',
 'resources_delete',
 'resources_get',
 'resources_list']

##  Unregistering MCP Toolgroups

If a toolgroup (e.g., `mcp::openshift`) already exists, we can unregister it to avoid conflicts during registration. 

In [42]:
client.toolgroups.unregister(toolgroup_id="mcp::openshift") 

Now if you would like to check whether mcp::ansible is present in registered tool group.

In [43]:
toolgroups = client.tools.list()
registered_toolgroups = set(t.toolgroup_id for t in toolgroups)

print("Registered Toolgroups:")
for tg in registered_toolgroups:
    print(f" - {tg}")

Registered Toolgroups:
 - builtin::websearch
 - mcp::custom_tool
 - mcp::github
 - builtin::code_interpreter
 - builtin::rag
 - mcp::ansible


There you see, you do not find openshift in registered tool group.

## Register an MCP tool group

We now register a new MCP toolgroup using `client.toolgroups.register(...)`. Ensure the endpoint is valid. 

In [44]:
# Register MCP tools

mcp_ocp_url = os.getenv("OCP_MCP_SERVER_URL")
client.toolgroups.register(
    toolgroup_id="mcp::openshift",
    provider_id="model-context-protocol",
    mcp_endpoint={"uri":mcp_ocp_url}
)

### Verify Registered tools

Once a new MCP toolgroup is registered, we can confirm its presence by listing all currently registered toolgroups.

In [45]:
toolgroups = client.tools.list()
registered_toolgroups = set(t.toolgroup_id for t in toolgroups)

print("Registered Toolgroups:")
for tg in registered_toolgroups:
    print(f" - {tg}")

Registered Toolgroups:
 - mcp::openshift
 - builtin::websearch
 - mcp::custom_tool
 - mcp::github
 - builtin::code_interpreter
 - builtin::rag
 - mcp::ansible


## Summary

In this notebook, we explored how to manage Model Context Protocol (MCP) toolgroups within the LlamaStack ecosystem. Here's what we accomplished step-by-step:

- **Connected to a LlamaStack Server** using environment variables (`REMOTE_BASE_URL`) to enable interaction via `llama-stack-client`.

- **Inspected Registered Toolgroups** to understand which tool integrations were already available. This helps avoid redundant registrations.

- **Queried Available Tools** within a toolgroup (like `mcp::openshift`), giving insight into the tool APIs exposed by each MCP server.

- **Unregistered Toolgroups** safely when necessary — for example, before re-registering with a corrected endpoint or configuration.

- **Registered New Toolgroups** by providing a `toolgroup_id`, the MCP provider type (`model-context-protocol`), and the server URI (`ANSIBLE_MCP_SERVER_URL`). This showed how to onboard a new tool endpoint to your LlamaStack.

- **Verified Tools** after registration to confirm the toolgroup was correctly integrated and available to LLM agents.


For more details, refer to the [LlamaStack documentation](https://llama-stack.readthedocs.io/en/latest/building_applications) and [llama-stack-demos](https://github.com/opendatahub-io/llama-stack-demos).

